# Kashmiri Whisper v5 — licensed real-speech continuation
Warm-starts from v4 and trains only on provenance-preserving OpenSLR/AI4Bharat speech. Production is not modified by this notebook.

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q 'transformers==4.48.3' 'peft==0.14.0' 'accelerate==1.3.0' 'datasets==3.2.0' 'evaluate>=0.4' soundfile librosa jiwer huggingface_hub pyarrow
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    print('HF_TOKEN unavailable: using public OpenSLR only; gated AI4Bharat will be skipped.')
include_gated = '--include-gated' if os.getenv('HF_TOKEN') else ''
!python /kaggle/working/cipher/scripts/prepare-public-kashmiri-asr.py --output /kaggle/working/v5-licensed-speech --max-per-source 6000 {include_gated}
import pandas as pd
manifest = pd.read_csv('/kaggle/working/v5-licensed-speech/metadata.csv')
print(manifest.groupby(['source','license']).agg(clips=('filename','size'), hours=('duration',lambda x: x.sum()/3600)))
assert len(manifest) >= 1500, 'Insufficient verified real speech; refusing to train'

In [ ]:
import os, subprocess, sys
env = os.environ.copy()
env.update({
  'STAGE1_ADAPTER':'Sameer0313/kashmiri-whisper-lora-v4-candidate',
  'NUM_EPOCHS':'4','BATCH_SIZE':'2','GRAD_ACCUM':'8','LEARNING_RATE':'1e-5',
  'EVAL_SIZE':'200','SAVE_STEPS':'200',
  'AUDIO_ROOTS_ONLY':'1','REQUIRE_REAL_AUDIO':'1',
  'EXTRA_AUDIO_ROOTS':'/kaggle/working/v5-licensed-speech',
  'OUTPUT_DIR':'/kaggle/working/kashmiri-whisper-v5-candidate',
  'CHECKPOINTS_DIR':'/kaggle/working/whisper-v5-checkpoints',
  'HF_REPO_ID':'Sameer0313/kashmiri-whisper-lora-v5-candidate'
})
subprocess.run([sys.executable,'scripts/finetune-whisper-ultra-accuracy.py'],cwd='/kaggle/working/cipher',env=env,check=True)

In [ ]:
from pathlib import Path
import json, shutil
manifest = Path('/kaggle/working/v5-licensed-speech/metadata.csv')
report = {'candidate':'Sameer0313/kashmiri-whisper-lora-v5-candidate','training_manifest_rows':sum(1 for _ in manifest.open())-1,'production_promoted':False,'requires_controlled_v4_v5_evaluation':True}
Path('/kaggle/working/whisper-v5-training-report.json').write_text(json.dumps(report,indent=2))
shutil.make_archive('/kaggle/working/kashmiri-whisper-v5-candidate','zip','/kaggle/working/kashmiri-whisper-v5-candidate')
print(report)